In [2]:
import pandas as pd
import numpy as np
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from nltk.stem import PorterStemmer

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Sujoy\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Sujoy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
# Load the JSON data
dataframe = pd.read_json(r"D:\YouTube-Likes-predictor\JSONS\apnacollege.json")

# Normalize the 'videos' field in the JSON
dataframe = pd.json_normalize(dataframe['videos'])

In [4]:
dataframe.head()


,video title,views,days ago,products,likes
0,Docker Tutorial for beginners - Complete One Shot,"101,226",5,0,4.3K
1,How to prepare for Placements & Internships as...,"79,948",9,0,3.9K
2,How this Core branch student with just 6.5 CGP...,"76,262",12,0,2.9K
3,How this student cracked DE Shaw & Co ? Interv...,"61,667",20,0,2.2K
4,Don't Chase Everything | IIT Guwahati Session ...,"207,142",23,0,7.2K


In [5]:
def convert_likes(column):
    if column.endswith('K'):
        return float(column[:-1]) * 1_000  
    elif column.endswith('M'):
        return float(column[:-1]) * 1_000_000  
    else:
        return float(column)  


In [6]:
dataframe['likes'] = dataframe['likes'].apply(convert_likes)

In [7]:
dataframe['views'] = dataframe['views'].str.replace(',', '').astype(int)

In [8]:
dataframe

,video title,views,days ago,products,likes
0,Docker Tutorial for beginners - Complete One Shot,101226,5,0,4300.0
1,How to prepare for Placements & Internships as...,79948,9,0,3900.0
2,How this Core branch student with just 6.5 CGP...,76262,12,0,2900.0
3,How this student cracked DE Shaw & Co ? Interv...,61667,20,0,2200.0
4,Don't Chase Everything | IIT Guwahati Session ...,207142,23,0,7200.0
...,...,...,...,...,...
247,How to Start Web Development? Complete Roadmap...,3717334,1043,0,167000.0
248,Why do you Procrastinate? 5 Steps to BEAT IT.,751406,1048,0,49000.0
249,How to make money from Coding? 5 ways to earn ...,2145802,1051,0,96000.0
250,React Native vs Flutter | Which one should you...,661522,1055,0,18000.0


In [11]:
df=dataframe

In [14]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=100)

# Fit and transform the video titles
tfidf_matrix = tfidf_vectorizer.fit_transform(df['video title'])

# Convert TF-IDF matrix to a DataFrame
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

# Step 2: Feature Engineering - Extract word count from titles
df['word_count'] = df['video title'].apply(lambda x: len(x.split()))

# Combine the new features with the TF-IDF representation
features_df = pd.concat([df[['word_count']], tfidf_df], axis=1)


In [15]:
features_df

,word_count,10,2023,2024,2025,algorithm,algorithms,alpha,amazon,apnacollegeofficial,...,sum,surprise,tech,tier,time,tips,tutorial,vs,web,year
0,8,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.49317,0.000000,0.000000,0.0
1,13,0.0,0.0,0.0,0.62882,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0
2,16,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.501996,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0
3,14,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0
4,10,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,12,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.000000,0.439683,0.0
248,9,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0
249,15,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0
250,13,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.707107,0.000000,0.0


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Step 3: Define Target Variables (Likes & Views)
targets = ['likes', 'views']

# Store predictions
predictions = {}

for target in targets:
    # Splitting data into training and testing sets (80% train, 20% test)
    X_train, X_test, y_train, y_test = train_test_split(features_df, df[target], test_size=0.2, random_state=42)

    # Initialize and train a Random Forest Regressor
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # Predict on test data
    y_pred = model.predict(X_test)

    # Evaluate the model
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Store results
    predictions[target] = {'model': model, 'mae': mae, 'r2': r2, 'predictions': y_pred}

# Display the evaluation metrics
evaluation_df = pd.DataFrame({
    "Target": ["Likes", "Views"],
    "Mean Absolute Error": [predictions['likes']['mae'], predictions['views']['mae']],
    "R² Score": [predictions['likes']['r2'], predictions['views']['r2']]
})


In [ ]:
# Step 1: Extract keywords using CountVectorizer (Bag-of-Words)
vectorizer = CountVectorizer(stop_words='english', max_features=20)  # Extract top 20 keywords
keywords_matrix = vectorizer.fit_transform(df['video title'])

# Convert to DataFrame
keywords_df = pd.DataFrame(keywords_matrix.toarray(), columns=vectorizer.get_feature_names_out())


In [21]:
combined_df = pd.concat([features_df, keywords_df], axis=1)

In [24]:
combined_df

,word_count,10,2023,2024,2025,algorithm,algorithms,alpha,amazon,apnacollegeofficial,...,java,ma,new,placement,problem,series,shradha,student,tech,web
0,8,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,13,0.0,0.0,0.0,0.62882,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,16,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,...,0,0,0,1,0,0,0,1,1,0
3,14,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,...,0,1,0,0,0,0,1,1,0,0
4,10,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,...,0,1,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,12,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,1
248,9,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
249,15,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
250,13,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
